### LLaMA inference using RAG

This document will explain the inference using LLaMA by using GPT-4o answers as the RAG.

For RAG, we will use Chroma database. We will use Chunking of 50 words with overlap of 5 words. But complete sentences will be preserved. For the knowledgebase, the answers from both the training and test data of Gpt-4o-mini will be used.

For testing, we will only perform the evaluation using the test data

In [1]:
import os

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
import pandas as pd
import numpy as np
import pickle
import evaluate
import json
import chromadb
from chromadb.config import Settings
from langchain_text_splitters import RecursiveCharacterTextSplitter

/data/mn27889/miniconda3/envs/mental-health-agents/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Enabling progress_apply for pandas

In [4]:
tqdm.pandas()

### Reading the responses of GPT-4o-mini from test and train data

Train Data

In [5]:
ques_list = []
ans_list = []
llama_resp_list = []
gpt_resp_list = []

with open('phase2_data_kabatubare/train_kabatubare.jsonl', 'rb') as file:
    for line in file:
        json_object = json.loads(line)
        ques_list.append(json_object['question'])
        ans_list.append(json_object['answer'])
        llama_resp_list.append(json_object['llama_response_base'])
        gpt_resp_list.append(json_object['gpt_response_base'])
        
train_dataset = pd.DataFrame({'question': ques_list,
                          'answer': ans_list,
                          'llama_response_base': llama_resp_list,
                          'gpt_response_base': gpt_resp_list})
train_dataset

,question,answer,llama_response_base,gpt_response_base
0,i have a small dull ache in my left testicle a...,just give it a rest my friend. it is not likel...,"I'm here to provide you with information, but ...",It's not uncommon for individuals to experienc...
1,i've heard conflicting opinions. 7 weeks of pr...,hi the best thing is to live your life as norm...,Congratulations on the new addition to your fa...,"During pregnancy, there are several dietary re..."
2,my friend slept over that had fever blisters. ...,hi found you this little piece of info. oral h...,I can sense your concern and worry about the s...,I understand that you're feeling very anxious ...
3,what are some common food triggers for migraines?,you can use information from your diary and tr...,"As a medical knowledge assistant, I can provid...",Migraines can be triggered by a variety of fac...
4,why does grey hair itch so much? . why does my...,hair doesn't itch. scalps will itch. there are...,The itchy sensation associated with grey hair ...,Itching in gray hair can be attributed to seve...
...,...,...,...,...
18744,how to make money online? . a brand new approa...,please do not post advertising on webmd answer...,There are many legitimate ways to make money o...,I'm here to assist with health-related inquiri...
18745,what can i eat with the stomach flu? . i can't...,the best foods to eat while feeling nausea are...,"I'm not a doctor, but I can provide some gener...",When you're dealing with stomach flu (viral ga...
18746,what exams and tests help doctors to evaluate ...,doctors can often easily recognize ringworm wh...,"Ringworm of the scalp, also known as tinea cap...",To evaluate or test individuals for ringworm o...
18747,could i be pregnant?,the only way to know for sure you're pregnant ...,"I'm not a medical professional, but I can prov...",Whether you could be pregnant depends on sever...


Test Data

In [6]:
ques_list = []
ans_list = []
llama_resp_list = []
gpt_resp_list = []

with open('phase2_data_kabatubare/test_kabatubare.jsonl', 'rb') as file:
    for line in file:
        json_object = json.loads(line)
        ques_list.append(json_object['question'])
        ans_list.append(json_object['answer'])
        llama_resp_list.append(json_object['llama_response_base'])
        gpt_resp_list.append(json_object['gpt_response_base'])
        
test_dataset = pd.DataFrame({'question': ques_list,
                          'answer': ans_list,
                          'llama_response_base': llama_resp_list,
                          'gpt_response_base': gpt_resp_list})
test_dataset

,question,answer,llama_response_base,gpt_response_base
0,i don't have periods due to taking nuvaring. h...,you really shouldn't worry about getting pregn...,I'm happy to help you with your concern. Since...,If you're using NuvaRing and have missed it by...
1,when you can't digest food. my daughter is 38 ...,i'm sorry your daughter is having a hard time....,I'm so sorry to hear that your daughter is exp...,I'm sorry to hear about your daughter's strugg...
2,8 wks post-hemorrhoidectomy. knife-like pain o...,i'm sorry you're going through this. hemorrhoi...,"I'm not a doctor, but I can try to provide som...",I'm sorry to hear that you're experiencing the...
3,i sometimes feel a mild discomfort under my le...,after reading your full statement i would also...,"I'm not a medical professional, but I can try ...","I'm not a doctor, but I can offer some general..."
4,i had sex about 12 hours ago and i'm noticing ...,hi ok if your shaved and this is on your pubic...,"I'm not a medical professional, but I can prov...",It's good that you're paying attention to chan...
...,...,...,...,...
4683,bright blood in stool for a while now burning ...,hi but as your not really well now i still thi...,"I'm not a doctor, but I can provide some gener...","I'm not a doctor, but the symptoms you're desc..."
4684,i feel fine i'm in a great mood but food taste...,well i'm happy you're in a good mood anyway! i...,"I'm glad you're feeling fine overall, but the ...",It's good to hear that you're feeling fine and...
4685,i have recently had unprotected sex and both m...,"you had unprotected sex? it is your ""other"" he...",I'm here to provide you with information and g...,Experiencing irritation on the head of the pen...
4686,is there a coated naproxen medication to preve...,all over-the-counter and prescription-strength...,"Yes, there are coated naproxen medications ava...","Yes, there are formulations of naproxen that a..."


Combining both the test and train data into a single dataframe

In [7]:
comp_dataset = pd.concat([train_dataset, test_dataset], axis=0).reset_index(drop=True)
comp_dataset

,question,answer,llama_response_base,gpt_response_base
0,i have a small dull ache in my left testicle a...,just give it a rest my friend. it is not likel...,"I'm here to provide you with information, but ...",It's not uncommon for individuals to experienc...
1,i've heard conflicting opinions. 7 weeks of pr...,hi the best thing is to live your life as norm...,Congratulations on the new addition to your fa...,"During pregnancy, there are several dietary re..."
2,my friend slept over that had fever blisters. ...,hi found you this little piece of info. oral h...,I can sense your concern and worry about the s...,I understand that you're feeling very anxious ...
3,what are some common food triggers for migraines?,you can use information from your diary and tr...,"As a medical knowledge assistant, I can provid...",Migraines can be triggered by a variety of fac...
4,why does grey hair itch so much? . why does my...,hair doesn't itch. scalps will itch. there are...,The itchy sensation associated with grey hair ...,Itching in gray hair can be attributed to seve...
...,...,...,...,...
23432,bright blood in stool for a while now burning ...,hi but as your not really well now i still thi...,"I'm not a doctor, but I can provide some gener...","I'm not a doctor, but the symptoms you're desc..."
23433,i feel fine i'm in a great mood but food taste...,well i'm happy you're in a good mood anyway! i...,"I'm glad you're feeling fine overall, but the ...",It's good to hear that you're feeling fine and...
23434,i have recently had unprotected sex and both m...,"you had unprotected sex? it is your ""other"" he...",I'm here to provide you with information and g...,Experiencing irritation on the head of the pen...
23435,is there a coated naproxen medication to preve...,all over-the-counter and prescription-strength...,"Yes, there are coated naproxen medications ava...","Yes, there are formulations of naproxen that a..."


### Making Chunks of the Input Data to be used for RAG

Let's analyze the distribution of characters in each gpt answer

In [8]:
comp_dataset['gpt_response_base'].apply(lambda x: len(x)).describe()

count    23437.000000
mean      1581.831506
std        599.059524
min          4.000000
25%       1134.000000
50%       1545.000000
75%       1970.000000
max       4259.000000
Name: gpt_response_base, dtype: float64

Let's use Chunk Size of 1000 and Overlap of 500. This will (on average) ensure that an answer gets divided into 3-4 chunks

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=500,
    length_function=len,
    is_separator_regex=False,
    separators=["\n\n", "\n", ".", " ",  ""],
)

Let's chunk the documents and save them into a unified list

In [10]:
gpt_response_chunks = []

for index, row in comp_dataset.iterrows():
    gpt_response_chunks = gpt_response_chunks + text_splitter.split_text(row['gpt_response_base'])

print(len(gpt_response_chunks))

62136


Let's generate the ids for each chunk

In [11]:
gpt_response_chunks_id = list(range(0, len(gpt_response_chunks)))
gpt_response_chunks_id = [str(id) for id in gpt_response_chunks_id]
len(gpt_response_chunks_id)

62136

### Setting up the Chroma Database

Create a persistent client to read the database from a defined directory

In [12]:
chromadb_client = chromadb.PersistentClient(path="./chroma_db")

Create or get the `kabatubare_dataset` collection

In [13]:
kabatubare_dataset_collection = chromadb_client.get_or_create_collection(name="kabatubare_dataset")

Chroma only allows a batch size of 5461 for addition. Let's prepare the batches with max size of 5000 and put them one by one into the collection

In [14]:
index_array = np.arange(start=0, stop=len(gpt_response_chunks), step=5000)
index_array = np.append(index_array, len(gpt_response_chunks))
index_array

array([    0,  5000, 10000, 15000, 20000, 25000, 30000, 35000, 40000,
       45000, 50000, 55000, 60000, 62136])

In [15]:
# for i in range(0, len(index_array)-1):
#     start = index_array[i]
#     end = index_array[i+1]
#     print(f"Adding {start} to {end}")
#     kabatubare_dataset_collection.add(
#         documents=gpt_response_chunks[start:end],
#         ids=gpt_response_chunks_id[start:end],
#     )

Let's see how many embeddings are stored in our collection

In [16]:
kabatubare_dataset_collection.count()

62136

Let's see the first five documents in our database

In [17]:
kabatubare_dataset_collection.get()['documents'][0:5]

["It's not uncommon for individuals to experience discomfort in the testicles after increased sexual activity, including frequent masturbation. Here are a few possibilities that could explain your symptoms:\n\n1. **Muscle strain**: Excessive stimulation can lead to muscle tension or strain in the pelvic area, potentially causing dull aches.\n\n2. **Refractory period**: After orgasm, it’s normal for the testicles and surrounding tissues to feel sensitive or ache due to the temporary changes in blood flow and hormone levels.\n\n3. **Epididymitis**: This is an inflammation of the epididymis (the tube that stores and carries sperm), which can sometimes occur after extensive sexual activity. Symptoms may include pain and swelling.\n\n4. **Testicular torsion or other acute conditions**: These are less likely, especially given that you mentioned no lumps or mass, but they can cause severe pain and require immediate medical attention.",
 "3. **Epididymitis**: This is an inflammation of the epi

Let's get some top-5 results for a dummy query to see if our system is working well

In [18]:
results = kabatubare_dataset_collection.query(
    query_texts=["What should I do if I have a headache?"],
    n_results=5
)

results

{'ids': [['31334', '11909', '23142', '61836', '48098']],
 'embeddings': None,
 'documents': [["It’s important to listen to your body. If the headache continues or worsens, or if you develop new symptoms (like fever, visual changes, nausea, or confusion), you should seek medical attention promptly. It's always better to err on the side of caution, especially when it comes to headaches or any potential reactions to substances like insect repellents.\n\nIf you decide to see a doctor, they may perform a physical examination and, if necessary, imaging or additional tests to help determine the cause of your symptoms. In the meantime, consider keeping hydrated, using a cold or warm pack on your neck, and practicing relaxation techniques to help alleviate tension.",
   '3. **Rest**: Ensure plenty of rest in a quiet and comfortable environment. Stress and fatigue can exacerbate headaches.\n\n4. **Cool Compress**: Applying a cool compress to the forehead or the back of the neck may help alleviat

In [19]:
results['documents'][0]

["It’s important to listen to your body. If the headache continues or worsens, or if you develop new symptoms (like fever, visual changes, nausea, or confusion), you should seek medical attention promptly. It's always better to err on the side of caution, especially when it comes to headaches or any potential reactions to substances like insect repellents.\n\nIf you decide to see a doctor, they may perform a physical examination and, if necessary, imaging or additional tests to help determine the cause of your symptoms. In the meantime, consider keeping hydrated, using a cold or warm pack on your neck, and practicing relaxation techniques to help alleviate tension.",
 '3. **Rest**: Ensure plenty of rest in a quiet and comfortable environment. Stress and fatigue can exacerbate headaches.\n\n4. **Cool Compress**: Applying a cool compress to the forehead or the back of the neck may help alleviate headache symptoms.\n\n5. **Caffeine**: In some cases, small amounts of caffeine can relieve h

Now that our Chroma database is setup, all the chunks are inserted and the query is working well. It's time that we initiated the LLaMA Model and pass these chunks along with each query to see the performance of LLaMA

### Formatting the questions

For each question in test dataset, we need to form a question in the form of rag string which needs to be passed to model. For this we need to extract the context from Chroma Database and then put it in the form of RAG sring which will be passed as user query to the model

Defining the function to get context related to query

In [20]:
def get_context_query(question):
    # get top-5 results from the database
    query_context = kabatubare_dataset_collection.query(
        query_texts=[question],
        n_results=5
    )

    query_context = query_context['documents'][0]

    query_context_str = ""
    for context in query_context:
        query_context_str = query_context_str + context + "\n\n"
    
    return query_context_str

Specifying the rag query format and formatting the query in this format

In [21]:
rag_query_str = '''Answer the QUESTION below by utilizing the information highlighted in the CONTEXT relevant to the QUESTION.
The final ANSWER should be a complete and coherent response to the QUESTION and should not include any of the CONTEXT or QUESTION text.
-----------------
CONTEXT: {context}
-----------------
QUESTION: {query}
-----------------
ANSWER: 
'''

In [22]:
def data_questions_formatted(question):
    query_context_str = get_context_query(question)
    
    rag_query_str_formatted = rag_query_str.format(context=query_context_str, query=question)
    
    return rag_query_str_formatted

Generating the context and applying the formatting. This took around 15 minutes, so the result is saved and will be loaded from there

In [23]:
# test_dataset_questions = test_dataset['question']
# test_dataset_questions_formatted_rag = test_dataset_questions.progress_apply(data_questions_formatted)

# with open('phase4_kabatubare_medical/test_dataset_questions_formatted_rag.pkl', 'wb') as file:
#     pickle.dump(test_dataset_questions_formatted_rag, file)

In [24]:
with open('phase4_kabatubare_medical/test_dataset_questions_formatted_rag.pkl', 'rb') as file:
    test_dataset_questions_formatted_rag = pickle.load(file)

### Loading the LLaMA Model

In [25]:
with open('../hf_token.key', 'r') as f:
    hf_token = f.read()

model_id = "meta-llama/Llama-3.2-3B-Instruct"

Loading the model shards

In [26]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
tokenizer.model_max_length = 4096

Loading the model shards

In [27]:
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto") # Must be float32 for MacBooks!
model.config.pad_token_id = tokenizer.pad_token_id # Updating the model config to use the special pad token

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.01s/it]


Defining the `eos` token as the terminator to finish the sentences

In [28]:
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

Defining the function to get the LLaMA Responses

In [29]:
def get_llama_response(question_inputs: str):
    
    llama_inputs = [[{"role": "system", "content": "You are a medical knowledge assistant trained to provide information and guidance on various health-related topics."},
                     {"role": "user", "content": question}] for question in question_inputs]

    texts = tokenizer.apply_chat_template(llama_inputs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(texts, padding="longest", truncation=True, return_tensors="pt")
    inputs = {key: val.to(model.device) for key, val in inputs.items()}
    temp_texts = tokenizer.batch_decode(inputs['input_ids'], skip_special_tokens=True)
    
    gen_tokens = model.generate(
        **inputs, 
        max_new_tokens=4096, 
        pad_token_id=tokenizer.pad_token_id, 
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.7,
        # top_p=0.9
    )

    gen_text = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
    gen_text = [i[len(temp_texts[idx]):] for idx, i in enumerate(gen_text)]
    
    return gen_text

Preparing the batches of the data

In [30]:
batch_size = 2
dataset_questions = test_dataset['question']
dataset_answers = test_dataset['answer']
dataset_questions_formatted = test_dataset_questions_formatted_rag
batch_indices = np.arange(0, len(dataset_questions), batch_size)
if batch_indices[-1] != len(dataset_questions):
    batch_indices = np.append(batch_indices, len(dataset_questions))

Running the batch inference to get responses from LLaMA

In [31]:
# question_list = []
# answer_list = []
# llama_responses = []
# for i in tqdm(range(0, len(batch_indices) - 1)):
#     questions_input = dataset_questions[batch_indices[i]:batch_indices[i+1]]
#     orig_answers = dataset_answers[batch_indices[i]:batch_indices[i+1]]
#     questions_formatted_input = dataset_questions_formatted[batch_indices[i]:batch_indices[i+1]]
#     llama_resp = get_llama_response(questions_formatted_input)
    
#     question_list = question_list + questions_input.to_list()
#     answer_list = answer_list + orig_answers.to_list()
#     llama_responses = llama_responses + llama_resp

# with open('phase4_kabatubare_medical/questions.pkl', 'wb') as file:
#     pickle.dump(question_list, file)
    
# with open('phase4_kabatubare_medical/answers.pkl', 'wb') as file:
#     pickle.dump(answer_list, file)

# with open('phase4_kabatubare_medical/llama_resp.pkl', 'wb') as file:
#     pickle.dump(llama_responses, file)

### Evaluaing the LLaMA Responses

Reading the questions, answers and LLaMA Responses for

In [32]:
with open('phase4_kabatubare_medical/questions.pkl', 'rb') as file:
    question_list = pickle.load(file)
    
with open('phase4_kabatubare_medical/answers.pkl', 'rb') as file:
    answer_list = pickle.load(file)

with open('phase4_kabatubare_medical/llama_resp.pkl', 'rb') as file:
    llama_responses = pickle.load(file)

In [33]:
question_list[0:10]

["i don't have periods due to taking nuvaring. how would i know if i was pregnant since i don't have any periods to miss? . the last two times i put the nuvaring in i put it in a day late. i have pregnancy symptoms and am wondering how soon i can take a pregnancy test for it to be accurate. i know you normally would take one several days after a missed period but i don't have periods.",
 "when you can't digest food. my daughter is 38 and when she eats the food just sits in her stomach and rots making her sick. she can't hardly eat anything yet her weight still climbs. she had her gallbladder removed several years ago and has had this problem ever since. dr. put her on reglin (she is off that now) but all it did was make her feel like she had chronic fatigue syndrome. help.",
 "8 wks post-hemorrhoidectomy. knife-like pain on b. m. too frequent b. m. anemic-still blood in stool on good diet/fiber. still have blood in stool and with bm. burning knife-like pain on bm. stools r soft or loos

In [34]:
answer_list[0:10]

["you really shouldn't worry about getting pregnanat while on birth control. your periods are being missed because of the birth control however that is completely normal. i would not be concerned with it at all",
 "i'm sorry your daughter is having a hard time. i think it's best for her to get checked out by a gastroenterologist to rule out any underlying problems. you can also have her see a registered dietitian to check for food intolerances and diet concerns related to gallbladder removal. here are some other things that might help: eat smaller more frequent meals. avoid high fat foods such as high fat meats full-fat dairy and fried foods. keep portions of fat (oil butter nuts/seeds) small at meals. get adequate fiber through fruits vegetables and whole grains. if fiber intake has been low gradually increase it. watch for individual items that are hard to digest such as dairy products and beverages like caffeine. have your daughter keep a journal to note which foods bother her (and 

In [35]:
llama_responses[0:10]

["If you're using NuvaRing and have missed it by a day a couple of times, it's understandable to be concerned about the possibility of pregnancy. Since you don't have regular periods while using hormonal contraception like NuvaRing, it can complicate understanding signs of pregnancy. \n\nTo determine if you're pregnant, you can take a home pregnancy test. For the most accurate result, it's generally recommended to wait at least a week after your missed period. However, since you don't have periods, you can take a pregnancy test as soon as you suspect you might be pregnant, but it's best to wait at least a week after the last time you had sex while using NuvaRing. \n\nKeep in mind that home pregnancy tests are most accurate when taken after a missed period, typically about 14 days after conception. Since you don't have periods, it's essential to consult a healthcare provider for guidance on the best time to take a pregnancy test and to discuss any concerns about your reproductive health

### Calculating the BLEU Score

In [36]:
bleu_eval = evaluate.load("bleu")
bleu_results = bleu_eval.compute(predictions=llama_responses, references=answer_list)
bleu_results

{'bleu': 0.014839747585239442,
 'precisions': [0.18641252802678335,
  0.025962690825557532,
  0.005832906217089278,
  0.0017178936095845077],
 'brevity_penalty': 1.0,
 'length_ratio': 1.3371639299232647,
 'translation_length': 686397,
 'reference_length': 513323}

### Calculating the ROUGE Score

In [37]:
rouge_eval = evaluate.load("rouge")
rouge_results = rouge_eval.compute(predictions=llama_responses, references=answer_list)
rouge_results

{'rouge1': np.float64(0.21963687293431416),
 'rouge2': np.float64(0.03944759827752637),
 'rougeL': np.float64(0.12723713951727206),
 'rougeLsum': np.float64(0.13632344682345307)}